# 02 – Collect party strengths per municipality

**Goal:** get the party strengths (in %) of every Swiss municipality in the
National Council elections (Nationalratswahlen) of **2019** and **2023**.

**Why we need this:** in our research question the party strengths are the
*explanation*. We later use them to predict the Yes-share of each municipality
in federal votes on environment, climate and energy since 2020.

**Steps in this notebook**

1. Ask the opendata.swiss Web API where the election files are.
2. Download the files and save them unchanged in `data/raw/parties/`.
3. Clean the data and group the parties into SVP, SP, FDP, Mitte, GPS, GLP and Other.
4. Build a *long* table and a *wide* table.
5. Check the data quality.
6. Save the tables in `data/processed/`.

**Output files**

- `data/processed/parties_municipal_long.csv`
- `data/processed/parties_municipal_wide.csv`

## 1. Load the libraries

We only need two libraries:

- **requests** sends requests to a Web API and downloads files from the internet.
- **pandas** works with tables (a bit like Excel, but in Python).

`os` is part of Python itself. We use it to create folders.

In [ ]:
import os

import pandas as pd
import requests

## 2. Define the folders

The notebook runs inside the `notebooks/` folder, so `..` means "go up one
folder" to the main project folder.

- Raw files go into `data/raw/parties/`. We never change them by hand.
- Cleaned tables go into `data/processed/`.

`os.makedirs(..., exist_ok=True)` creates the folder if it does not exist yet.

In [ ]:
RAW_DIR = "../data/raw/parties"
PROCESSED_DIR = "../data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw folder:      ", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

## 3. Ask the opendata.swiss Web API which files exist

opendata.swiss is the open data portal of Switzerland. It has a **Web API**
(called CKAN API). An API is an address that a program can ask for data. The
answer comes back as **JSON**, a text format that Python can read like a
dictionary.

We use the API action `package_show`. A *package* is a dataset. The Federal
Statistical Office (FSO) publishes one package per election:

- `eidg-wahlen-2019`
- `eidg-wahlen-2023`

Each package contains several *resources* (files). Here we print the list of
all files, so we can see what is available.

In [ ]:
API_URL = "https://ckan.opendata.swiss/api/3/action/package_show"

packages = {
    2019: "eidg-wahlen-2019",
    2023: "eidg-wahlen-2023",
}

package_info = {}

for year, package_name in packages.items():
    response = requests.get(API_URL, params={"id": package_name}, timeout=60)
    response.raise_for_status()  # stops with an error if the request failed
    package_info[year] = response.json()["result"]

    print(f"=== {package_name}: {len(package_info[year]['resources'])} files ===")
    for resource in package_info[year]["resources"]:
        title = resource["title"]["de"]  # the title exists in several languages
        print(f"  {resource['format']:<5} {title}")
    print()

## 4. Find the file with the party results per municipality

From the list above we need the file with the **party results**
("Ergebnisse Parteien") at **municipality level** (Gemeinden), in **CSV**
format (a simple table file).

- 2019: the file is called `NR - Ergebnisse Parteien (Gemeinden)`.
  There is also a file `(CH und Kantone)`, which only has Swiss and cantonal
  totals, so we skip it.
- 2023: the file is called `NR - Ergebnisse Parteien (csv)`. It contains all
  levels together (Switzerland, cantons and municipalities). We keep only the
  municipalities later.

The loop below goes through all files and keeps the one that matches these
rules. If no file is found, the notebook stops with a clear message.

In [ ]:
download_urls = {}

for year, info in package_info.items():
    for resource in info["resources"]:
        title = resource["title"]["de"]
        if (
            "Ergebnisse Parteien" in title
            and resource["format"] == "CSV"
            and "Kantone" not in title
        ):
            download_urls[year] = resource["url"]
            print(f"{year}: found '{title}'")
            print(f"      {resource['url']}")

    if year not in download_urls:
        raise ValueError(f"No municipal party results found for {year}. Please check manually.")

## 5. Download the files and save them unchanged

We download each file with `requests.get` and write the bytes exactly as they
arrive (`"wb"` = write binary). This way the raw file is an exact copy of the
original, and anybody can check our work later.

In [ ]:
raw_files = {}

for year, url in download_urls.items():
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    file_path = f"{RAW_DIR}/nr_{year}_parteien_gemeinden.csv"
    with open(file_path, "wb") as file:
        file.write(response.content)

    raw_files[year] = file_path
    print(f"{year}: saved {len(response.content) / 1_000_000:.1f} MB to {file_path}")

## 6. Read the 2019 file

The FSO files use a **semicolon** (`;`) between columns, not a comma.
`encoding="utf-8-sig"` makes sure special characters like ü, é and è are read
correctly.

Each row is **one party in one municipality**. The important columns are:

| Column | Meaning |
| --- | --- |
| `gemeinde_nummer` | FSO municipality number (our join key) |
| `gemeinde_bezeichnung` | municipality name |
| `kanton_nummer` | canton number (1 = Zurich ... 26 = Jura) |
| `partei_bezeichnung_de` | party name |
| `partei_staerke` | party strength in % |

In [ ]:
raw_2019 = pd.read_csv(raw_files[2019], sep=";", encoding="utf-8-sig")

print("Rows and columns:", raw_2019.shape)
raw_2019.head()

## 7. Read the 2023 file and keep only the municipalities

The 2023 file has a column `ebene_resultat` (result level). It tells us if a
row is about all of Switzerland (`Schweiz`), a canton (`Kanton`) or a
municipality (`Gemeinde`). We only keep the municipality rows.

In [ ]:
raw_2023 = pd.read_csv(raw_files[2023], sep=";", encoding="utf-8-sig")

print("Rows per level in the 2023 file:")
print(raw_2023["ebene_resultat"].value_counts())

raw_2023 = raw_2023[raw_2023["ebene_resultat"] == "Gemeinde"]
print()
print("Rows after keeping only municipalities:", raw_2023.shape)

## 8. Put both years into one table with English column names

Both files use the same German column names for the information we need. So
we can:

1. pick the five columns we need from each year,
2. add a column `year`,
3. stack the two tables on top of each other (`pd.concat`),
4. rename the columns to English.

`party_original` keeps the original party name from the FSO. We group the
parties in a later step.

In [ ]:
columns = [
    "gemeinde_nummer",
    "gemeinde_bezeichnung",
    "kanton_nummer",
    "partei_bezeichnung_de",
    "partei_staerke",
]

part_2019 = raw_2019[columns].copy()
part_2019["year"] = 2019

part_2023 = raw_2023[columns].copy()
part_2023["year"] = 2023

parties = pd.concat([part_2019, part_2023], ignore_index=True)

parties = parties.rename(columns={
    "gemeinde_nummer": "bfs_nr",
    "gemeinde_bezeichnung": "municipality",
    "kanton_nummer": "canton_nr",
    "partei_bezeichnung_de": "party_original",
    "partei_staerke": "party_strength_pct",
})

# The numbers should be whole numbers (in 2023 they were read as 1.0, 2.0, ...)
parties["bfs_nr"] = parties["bfs_nr"].astype(int)
parties["canton_nr"] = parties["canton_nr"].astype(int)

print("Rows:", len(parties))
parties.head()

## 9. Remove the "Swiss abroad" results

Swiss citizens living abroad can vote too. In some cantons their votes are
counted in an extra *pseudo-municipality*, for example "ZH-Ausland-CH".
These are not real places, so they do not fit our analysis of municipalities.

They are easy to recognise by their number:

- 2019: numbers from 9000 upwards (e.g. 9010)
- 2023: numbers from 19000 upwards (e.g. 19010)

Real municipality numbers are always below 9000, so we keep only those.

In [ ]:
abroad = parties[parties["bfs_nr"] >= 9000]
print("Removed 'Swiss abroad' units:")
print(abroad[["year", "bfs_nr", "municipality"]].drop_duplicates().to_string(index=False))

parties = parties[parties["bfs_nr"] < 9000]
print()
print("Rows left:", len(parties))

## 10. Add the canton abbreviation

The files only have a canton *number*. The FSO always uses the same order
(1 = ZH, 2 = BE, ... 26 = JU). With a small dictionary we translate the
number into the usual two-letter abbreviation, which is easier to read.

In [ ]:
canton_abbreviations = {
    1: "ZH", 2: "BE", 3: "LU", 4: "UR", 5: "SZ", 6: "OW", 7: "NW",
    8: "GL", 9: "ZG", 10: "FR", 11: "SO", 12: "BS", 13: "BL", 14: "SH",
    15: "AR", 16: "AI", 17: "SG", 18: "GR", 19: "AG", 20: "TG", 21: "TI",
    22: "VD", 23: "VS", 24: "NE", 25: "GE", 26: "JU",
}

parties["canton"] = parties["canton_nr"].map(canton_abbreviations)

print("Rows without a canton:", parties["canton"].isna().sum())
print(parties[["canton_nr", "canton"]].drop_duplicates().sort_values("canton_nr").to_string(index=False))

## 11. Empty party strengths mean 0 %

Some rows have no value in `party_strength_pct`. This happens when a party was
**not on the ballot** in that canton, so nobody could vote for it there. The
correct value is therefore **0 %**, not "unknown".

We first count the empty values, then fill them with 0.

In [ ]:
print("Empty party strengths per year (before filling):")
print(parties[parties["party_strength_pct"].isna()].groupby("year").size())

parties["party_strength_pct"] = parties["party_strength_pct"].fillna(0)

print()
print("Empty party strengths after filling:", parties["party_strength_pct"].isna().sum())

## 12. Group the parties

For our analysis we keep the six largest parties and put all smaller parties
into **Other**:

| Group | Party | 2019 name in FSO file | 2023 name in FSO file |
| --- | --- | --- | --- |
| SVP | Swiss People's Party | SVP | SVP |
| SP | Social Democrats | SP | SP |
| FDP | Liberals | FDP | FDP |
| **Mitte** | The Centre | **CVP + BDP** | Mitte |
| GPS | Greens | GPS | GRÜNE |
| GLP | Green Liberals | GLP | GLP |
| Other | all other parties | EVP, EDU, Lega, ... | EVP, EDU, Lega, ... |

**Why CVP + BDP = Mitte?** In 2019 the CVP (Christian Democrats) and the BDP
(Conservative Democrats) were still two separate parties. In 2021 they merged
into the new party **Die Mitte**. To compare 2019 with 2023 we must look at the
same party in both years. So for 2019 we **add the CVP and BDP strengths
together** and call the result "Mitte".

**Small notes:**

- The Greens are called "GPS" in 2019 and "GRÜNE" in 2023. It is the same
  party, so both become "GPS".
- "LPS" (Liberals, mainly the LDP in Basel-Stadt) is listed separately by the
  FSO and therefore goes into "Other", like in the official FSO tables.

The dictionary `party_groups` translates every original name into a group.
Every name that is not in the dictionary becomes "Other".

In [ ]:
party_groups = {
    "SVP": "SVP",
    "SP": "SP",
    "FDP": "FDP",
    "CVP": "Mitte",
    "BDP": "Mitte",
    "Mitte": "Mitte",
    "GPS": "GPS",
    "GRÜNE": "GPS",
    "GLP": "GLP",
}

parties["party"] = parties["party_original"].map(party_groups).fillna("Other")

# Show which original party ends up in which group
print(parties[["year", "party_original", "party"]].drop_duplicates().sort_values(["party", "year"]).to_string(index=False))

## 13. Add up the parties inside each group

Now we add up the strengths inside each group. For example, in a 2019
municipality: Mitte = CVP + BDP, and Other = EVP + EDU + ... .

`groupby(...).sum()` does exactly this: it takes all rows with the same year,
municipality and party group and adds up their percentages.

In [ ]:
grouped = (
    parties
    .groupby(["year", "bfs_nr", "municipality", "canton", "party"], as_index=False)["party_strength_pct"]
    .sum()
)

print("Rows:", len(grouped))
grouped.head(7)

## 14. Build the wide table

The **wide table** has one row per *municipality and year*, and one column
per party. This shape is what we need later for the regression (prediction
model), because there every party is one explanatory variable.

`pivot_table` turns the party names into columns.

**Important:** if a party did not run in a canton at all, the FSO file
sometimes has **no row** for it (not even an empty one). Example: the GLP did
not run in some small cantons. After the pivot, such a cell would be empty.
But the correct value is **0 %**, because nobody could vote for that party
there. `fill_value=0` writes 0 into these cells.

In [ ]:
party_order = ["SVP", "SP", "FDP", "Mitte", "GPS", "GLP", "Other"]

wide_table = grouped.pivot_table(
    index=["year", "bfs_nr", "municipality", "canton"],
    columns="party",
    values="party_strength_pct",
    fill_value=0,
).reset_index()

wide_table.columns.name = None  # remove the leftover label "party" above the columns
wide_table = wide_table[["year", "bfs_nr", "municipality", "canton"] + party_order]
wide_table[party_order] = wide_table[party_order].round(3)

print("Rows:", len(wide_table))
wide_table.head()

## 15. Build the long table

The **long table** has the same information, arranged differently:
one row per *year x municipality x party*. Long tables are practical for
filtering and for charts (e.g. "show me all SVP values").

`melt` is the opposite of `pivot_table`: it turns the party columns back into
rows. Because we start from the wide table, every municipality now has all
7 party groups (including the 0 % values).

In [ ]:
long_table = wide_table.melt(
    id_vars=["year", "bfs_nr", "municipality", "canton"],
    value_vars=party_order,
    var_name="party",
    value_name="party_strength_pct",
)

# Sort: year, then municipality, then parties in our fixed order
long_table["party_rank"] = long_table["party"].map({name: i for i, name in enumerate(party_order)})
long_table = long_table.sort_values(["year", "bfs_nr", "party_rank"]).drop(columns="party_rank")
long_table = long_table.reset_index(drop=True)

print("Rows:", len(long_table))
long_table.head(14)

## 16. Check 1: number of municipalities per year

Switzerland has fewer municipalities every year because small municipalities
merge. So we expect **fewer municipalities in 2023 than in 2019**.

We also count how many municipality numbers exist in only one of the two
years. These are mostly mergers. We do **not** fix this here; this happens
later in the integration step with the FSO municipality register. Here we just
document the numbers.

In [ ]:
print("Municipalities per year:")
print(wide_table.groupby("year")["bfs_nr"].nunique())

numbers_2019 = wide_table[wide_table["year"] == 2019]["bfs_nr"]
numbers_2023 = wide_table[wide_table["year"] == 2023]["bfs_nr"]

print()
print("Municipality numbers only in 2019:", (~numbers_2019.isin(numbers_2023)).sum())
print("Municipality numbers only in 2023:", (~numbers_2023.isin(numbers_2019)).sum())
print("Municipality numbers in both years:", numbers_2019.isin(numbers_2023).sum())

## 17. Check 2: do the party strengths add up to 100 %?

In every municipality, all parties together should make **100 %**. We add up
the seven party columns for each row of the wide table. Because of rounding,
we accept small differences (between 99.9 % and 100.1 %).

In [ ]:
wide_table["total_pct"] = wide_table[party_order].sum(axis=1)

print("Total per municipality (summary):")
print(wide_table.groupby("year")["total_pct"].describe()[["count", "min", "max"]])

not_100 = wide_table[(wide_table["total_pct"] < 99.9) | (wide_table["total_pct"] > 100.1)]
print()
print("Municipalities where the total is NOT about 100 %:", len(not_100))

# We only needed this column for the check, so we remove it again
wide_table = wide_table.drop(columns="total_pct")

## 18. Check 3: missing values and completeness

- Are there any empty cells left in the two tables?
- Does every municipality have exactly **7 rows** in the long table
  (one per party group)?

In [ ]:
print("Missing values in the long table:")
print(long_table.isna().sum())

print()
print("Missing values in the wide table:")
print(wide_table.isna().sum())

rows_per_municipality = long_table.groupby(["year", "bfs_nr"]).size()
print()
print("Municipalities that do not have exactly 7 party rows:", (rows_per_municipality != 7).sum())

## 19. Save the tables

We save both tables as CSV files in `data/processed/`. `index=False` means
pandas does not write its internal row numbers into the file.

In [ ]:
long_path = f"{PROCESSED_DIR}/parties_municipal_long.csv"
wide_path = f"{PROCESSED_DIR}/parties_municipal_wide.csv"

long_table.to_csv(long_path, index=False)
wide_table.to_csv(wide_path, index=False)

print("Saved", long_path, "-", len(long_table), "rows")
print("Saved", wide_path, "-", len(wide_table), "rows")

## Summary

- **Source:** FSO National Council elections 2019 and 2023, found and
  downloaded through the opendata.swiss CKAN Web API (`package_show`).
- **Raw files:** saved unchanged in `data/raw/parties/`.
- **Cleaning:** removed the "Swiss abroad" pseudo-municipalities, set missing
  party strengths to 0 % (party not on the ballot), grouped parties into
  SVP, SP, FDP, Mitte (2019: CVP + BDP), GPS, GLP and Other.
- **Outputs:** `parties_municipal_long.csv` and `parties_municipal_wide.csv`
  in `data/processed/`.
- **Not done yet:** municipal mergers. The 2019 data uses the municipalities of
  2019 and the 2023 data those of 2023. This is handled in a later
  integration step with the FSO municipality register.